<a href="https://colab.research.google.com/github/arabadzhyanzhela-collab/github-lab-2/blob/main/S08_Miniapp_09_Triaje_de_urgencias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Miniapp 9 · ¿Este paciente es urgente?

**Nivel:** 3/3 · **Duración:** 55 min · **Tema:** modelo explicable frente a modelo potente
**Practica:** S07 (umbral, recall, matriz de confusión) y S08 (árbol, bosque, boosting, elegir campeón)

---

## Objetivo
Construir una **ayuda para ordenar la sala de espera** de urgencias: a partir de los signos vitales de un paciente, el modelo estima cuánto se parece a un caso urgente.

Durante la práctica hay que tomar una decisión real:
- un modelo **que se pueda explicar a una enfermera** (árbol pequeño), o
- un modelo **más potente pero opaco** (bosque aleatorio o boosting).

> **Aviso importante:** es un ejercicio de aprendizaje con datos inventados. **Ninguna app de este tipo decide quién se atiende primero**: en urgencias reales decide siempre una persona.

## Datos
Fichero `triaje_urgencias.csv`: 4 000 pacientes **inventados**, con un **13 % de casos urgentes**.

| Columna | Significado |
|---|---|
| `edad` | años del paciente |
| `frecuencia_cardiaca` | pulsaciones por minuto |
| `tension_sistolica` | tensión arterial alta (mmHg) |
| `saturacion_o2` | oxígeno en sangre (%) |
| `temperatura_c` | temperatura corporal (°C) |
| `dolor_0_10` | dolor que declara el paciente (0 = nada, 10 = máximo) |
| `dolor_pecho` | 1 = tiene dolor de pecho |
| `dificultad_respirar` | 1 = le cuesta respirar |
| `llega_en_ambulancia` | 1 = llega en ambulancia |
| **`urgente`** | **objetivo**: 1 = necesitó atención inmediata |

## Índice
1. Preparar el entorno
2. Cargar y comparar los datos
3. Separar entrenamiento y examen
4. El duelo de modelos
5. El árbol explicado en palabras
6. El umbral pensando en la seguridad
7. El examen (una sola vez)
8. ¿Falla más con unos pacientes que con otros?
9. La app: ordenar la sala de espera
10. Ejercicios extra (con solución)
11. Hoja del equipo
12. Prueba
13. Conclusiones finales

**Cómo usar el cuaderno:** ejecuta las celdas en orden, de arriba abajo (`Mayús + Intro`). Si algo falla: *Entorno de ejecución → Reiniciar y ejecutar todo*.

---
## Paso 1 · Preparar el entorno

Importamos las librerías necesarias. Para qué sirve cada una:

- **pandas / numpy**: trabajar con tablas y hacer cálculos.
- **train_test_split**: separar los datos en entrenamiento y examen.
- **StratifiedKFold, cross_validate, cross_val_predict**: validación cruzada (evaluar el modelo varias veces con distintos trozos de los datos de entrenamiento).
- **Modelos**:
  - `DummyClassifier`: modelo «tonto» de referencia, no mira los datos.
  - `LogisticRegression`: modelo lineal clásico (necesita escalar los datos con `StandardScaler`).
  - `DecisionTreeClassifier`: árbol de decisión; si es pequeño, **se puede leer como reglas**.
  - `RandomForestClassifier`: bosque, es decir, cientos de árboles que votan. Más potente, pero opaco.
  - `HistGradientBoostingClassifier`: boosting, árboles que se corrigen unos a otros. También potente y opaco.
- **Métricas**: curva precisión–recall, matriz de confusión y AUC.

**Nota:** `warnings.filterwarnings("ignore")` oculta avisos que no afectan al resultado, para que la salida quede limpia.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np

from sklearn.model_selection import train_test_split, cross_validate, cross_val_predict, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import precision_recall_curve, confusion_matrix, roc_auc_score

print("Librerías cargadas correctamente")

---
## Paso 2 · Cargar y comparar los datos

Leemos el CSV desde GitHub y contamos cuántos pacientes son urgentes.

Después calculamos la **media de cada columna** separando no urgentes (`0`) y urgentes (`1`). Así vemos, **antes de entrenar ningún modelo**, qué signos vitales cambian más en los casos urgentes.

**Nota:** `df["urgente"].mean()` da directamente el porcentaje de urgentes, porque la columna solo contiene 0 y 1. El `.T` final gira la tabla para que cada fila sea una variable y se lea mejor.

In [ ]:
URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/triaje_urgencias.csv"
df = pd.read_csv(URL)

print("Filas y columnas:", df.shape)
print("Pacientes urgentes:", int(df["urgente"].sum()), f"({df['urgente'].mean():.1%})")
df.head()

In [ ]:
# Media de cada variable: no urgentes (0) frente a urgentes (1)
comparacion = df.groupby("urgente").mean().round(1).T
comparacion.columns = ["No urgente (0)", "Urgente (1)"]
comparacion

### Qué deberías ver
- **4 000 pacientes, 13 % urgentes** (529 pacientes).
- Los urgentes tienen **más dolor** (≈ 4,8 frente a 3,2), **pulso más alto** (≈ 92 frente a 85), **menos saturación de oxígeno** (≈ 95,6 % frente a 96,6 %) y más casos de **dificultad para respirar** y **dolor de pecho**.

### Conclusión del paso
Las clases están **desequilibradas**: casi 9 de cada 10 pacientes no son urgentes. Esto significa que la precisión global (*accuracy*) puede engañar, como ya vimos en la miniapp de fraude.

---
## Paso 3 · Separar entrenamiento y examen

- **Entrenamiento (75 %)**: con estos datos aprenden los modelos y elegimos el umbral.
- **Examen (25 %)**: se guarda y **no se mira** hasta el final. Se usa una sola vez.

**¿Por qué `stratify=y`?** Para que el examen tenga el mismo 13 % de urgentes que el conjunto completo. Sin esto, el reparto al azar podría dejar muy pocos urgentes en una de las partes.

**Validación cruzada (`cv`)**: dividimos el entrenamiento en 5 partes; el modelo se entrena con 4 y se evalúa con la 5.ª, rotando 5 veces. La nota final es la media. Así evaluamos sin tocar el examen.

In [ ]:
y = df["urgente"]
X = df.drop(columns="urgente")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=42)

print(len(X_tr), "pacientes para entrenar ·", len(X_te), "para el examen (NO mirar)")
print("Urgentes en entrenamiento:", int(y_tr.sum()), "· en el examen:", int(y_te.sum()))

### Qué deberías ver
**3 000** pacientes para entrenar y **1 000** para el examen, con **132 urgentes** en el examen.

---
## Paso 4 · El duelo de modelos

Comparamos cinco modelos con validación cruzada y tres métricas:

| Métrica | Qué mide | Valor de un modelo que no sabe nada |
|---|---|---|
| **Precisión global** (*accuracy*) | porcentaje de aciertos totales | ≈ 86,8 % (engaña) |
| **AUC** (área bajo la curva ROC) | si el modelo pone a los urgentes por encima de los no urgentes | 0,50 |
| **AP** (precisión media, curva precisión–recall) | calidad de los avisos centrándose solo en los urgentes | ≈ 0,13 |

Fíjate especialmente en el contraste entre:
- **Árbol explicable** (profundidad máxima 4, mínimo 20 pacientes por hoja): se puede leer y explicar.
- **Bosque** y **Boosting**: más potentes, pero no se pueden explicar con unas pocas reglas.

**Nota:** la logística va dentro de un `make_pipeline` con `StandardScaler` porque necesita las variables en la misma escala. Los árboles no lo necesitan.

Esta celda tarda unos segundos (el bosque tiene 300 árboles y se entrena 5 veces).

In [ ]:
modelos = {"Nadie es urgente (tonto)":         DummyClassifier(strategy="prior"),
           "Logística":                        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
           "Árbol explicable (profundidad 4)": DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42),
           "Bosque":                           RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=2),
           "Boosting":                         HistGradientBoostingClassifier(random_state=42)}

filas = {}
for nombre, m in modelos.items():
    r = cross_validate(m, X_tr, y_tr, cv=cv, scoring=["accuracy", "roc_auc", "average_precision"])
    filas[nombre] = {"precisión global": r["test_accuracy"].mean(),
                     "AUC":              r["test_roc_auc"].mean(),
                     "AP":               r["test_average_precision"].mean()}

tabla = pd.DataFrame(filas).T.round(3)
tabla

### Qué deberías ver
| Modelo | AUC | AP |
|---|---|---|
| Nadie es urgente (tonto) | 0,50 | ≈ 0,13 |
| Logística | ≈ 0,78 | ≈ 0,42 |
| Árbol explicable | ≈ 0,80 | ≈ 0,47 |
| **Bosque** | **≈ 0,87** | **≈ 0,64** |
| Boosting | ≈ 0,85 | ≈ 0,61 |

El modelo tonto acierta un **86,8 %** sin detectar a ningún urgente.

### Conclusión del paso
- La precisión global apenas distingue los modelos; **AUC y AP sí**.
- El **bosque** es el mejor en nota, seguido del boosting.
- El **árbol explicable** queda por detrás, pero muy por encima del tonto.

---
## Paso 5 · El árbol explicado en palabras

Entrenamos el árbol pequeño con todo el entrenamiento e imprimimos sus reglas con `export_text`.

**Cómo leer la salida:**
- Cada línea `|---` es una **pregunta** sobre una variable (por ejemplo, `dolor_0_10 <= 6.50`).
- Cuanto más a la derecha, más profunda es la pregunta.
- `class: 1` al final de una rama significa que esa combinación de respuestas lleva a **urgente**; `class: 0`, a **no urgente**.

Esto es lo que hace valioso al árbol: **se le puede leer en voz alta a una enfermera**.

In [ ]:
arbol = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42).fit(X_tr, y_tr)
print(export_text(arbol, feature_names=list(X.columns), max_depth=4))

In [ ]:
# Qué variables usa más el árbol (importancia: cuánto ayuda cada una a separar urgentes)
importancia = pd.Series(arbol.feature_importances_, index=X.columns).sort_values(ascending=False)
importancia[importancia > 0].round(3).to_frame("importancia en el árbol")

### Qué deberías ver
Una regla que se puede decir en voz alta, por ejemplo:

> *«Si el dolor es 6 o menos, el pulso 110 o menos, la saturación 93 o menos y le cuesta respirar → urgente.»*

Las variables más importantes son la **dificultad para respirar**, el **dolor**, el **pulso**, la **temperatura** y la **saturación de oxígeno**.

### Conclusión del paso
Con el árbol se puede **explicar** cada decisión y **auditar** si tiene sentido clínico. Con el bosque (300 árboles) esto no es posible.

---
## Paso 6 · El umbral pensando en la seguridad

En urgencias, **que se escape un paciente urgente es muy grave**. Por eso no usamos el umbral por defecto (0,50). Hacemos lo contrario:

1. **Fijamos el objetivo de seguridad**: detectar al menos al **90 % de los urgentes** (recall = 0,90).
2. Buscamos el umbral que lo consigue.
3. Miramos **cuántas falsas alarmas cuesta** con cada modelo.

**Conceptos:**
- **Recall (sensibilidad):** de todos los urgentes reales, cuántos detectamos.
- **Precisión de los avisos:** de todos los avisos, cuántos son urgentes de verdad. Si es baja, hay muchas falsas alarmas.

**Qué hace la función `umbral_para_recall`:**
1. Obtiene probabilidades «honestas» con `cross_val_predict` (cada paciente lo predice un modelo que no lo vio al entrenar).
2. Calcula la curva precisión–recall.
3. Se queda con el **umbral más alto que todavía cumple** el recall pedido, que es el que menos falsas alarmas da.

**Nota:** todo esto se hace **solo con los datos de entrenamiento**.

In [ ]:
def umbral_para_recall(modelo, recall_objetivo=0.90):
    proba = cross_val_predict(modelo, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
    precision, recall, umbrales = precision_recall_curve(y_tr, proba)
    i = np.where(recall[:-1] >= recall_objetivo)[0][-1]   # último umbral que aún cumple el recall
    return float(umbrales[i]), float(precision[i])

resultados_umbral = {}
for nombre in ["Árbol explicable (profundidad 4)", "Bosque", "Boosting"]:
    u, prec = umbral_para_recall(modelos[nombre])
    resultados_umbral[nombre] = (u, prec)
    print(f"{nombre:34s} umbral {u:.2f} · de cada 100 avisos, {prec * 100:.0f} son urgentes de verdad")

### Qué deberías ver
Exigiendo detectar al 90 % de los urgentes, de cada 100 avisos son urgentes de verdad:

| Modelo | Avisos correctos de cada 100 |
|---|---|
| Árbol explicable | ≈ 15 |
| **Bosque** | **≈ 24** |
| Boosting | ≈ 22 |

### Conclusión del paso
Esta es **la decisión central** de la práctica:
- El **bosque** genera **menos falsas alarmas** para el mismo nivel de seguridad (24 frente a 15 avisos útiles de cada 100).
- El **árbol** genera más falsas alarmas, pero **se puede explicar**.

No hay una respuesta única: depende de si el hospital valora más la eficiencia o la transparencia.

---
## Paso 7 · El examen: una sola vez, con el campeón y su umbral

Elegimos el campeón mirando las dos tablas anteriores (nota general y falsas alarmas). Por defecto usamos el **Bosque**; si vuestra tabla dice otra cosa, cambiad el nombre en `campeon_nombre`.

Pasos:
1. Calculamos el umbral del campeón para un recall del 90 % (solo con entrenamiento).
2. Entrenamos el campeón con todo el entrenamiento.
3. Predecimos el examen y aplicamos el umbral **sin modificarlo**.

**Cómo leer la matriz de confusión:**

| | Predice: en cola | Predice: prioridad |
|---|---|---|
| **Real: no urgente** | VN (bien clasificado) | FP (falsa alarma) |
| **Real: urgente** | FN (urgente que se escapa) | VP (urgente detectado) |

In [ ]:
campeon_nombre = "Bosque"                       # cámbialo si tu tabla dice otra cosa
campeon = modelos[campeon_nombre]

umbral, _ = umbral_para_recall(campeon)
final = campeon.fit(X_tr, y_tr)
proba = final.predict_proba(X_te)[:, 1]
aviso = proba >= umbral

vn, fp, fn, vp = confusion_matrix(y_te, aviso).ravel()
print(f"Campeón: {campeon_nombre} · umbral {umbral:.2f}")
print(f"Urgentes detectados {vp} de {vp + fn} · falsas alarmas {fp} de {vn + fp} no urgentes · AUC {roc_auc_score(y_te, proba):.3f}")

In [ ]:
# Matriz de confusión en forma de tabla
pd.DataFrame([[vn, fp], [fn, vp]],
             index=["Real: no urgente", "Real: urgente"],
             columns=["Predice: en cola", "Predice: prioridad"])

### Qué deberías ver
- Con el bosque se detectan **119 de 132 urgentes** (unos 90 %).
- A cambio, hay **364 falsas alarmas** de 868 no urgentes.
- AUC en el examen ≈ **0,89**.

### Conclusión del paso
El recall del 90 % que fijamos en entrenamiento **se mantiene en el examen**, así que el umbral generaliza bien. El precio es alto: unos 4 de cada 10 pacientes no urgentes reciben aviso. Es el coste de priorizar la seguridad.

**Cuidado:** solo hay 132 urgentes en el examen. Uno más o uno menos cambia los porcentajes, así que no hay que fiarse de los decimales.

---
## Paso 8 · ¿Falla más con unos pacientes que con otros?

Un modelo puede funcionar bien en promedio y mal en un grupo concreto. Comprobamos el recall (urgentes detectados) en dos grupos de edad:
- pacientes de **65 años o más**,
- pacientes **menores de 65**.

**Nota:** `max(urgentes.sum(), 1)` evita dividir entre cero si un grupo no tuviera urgentes.

In [ ]:
mayores = X_te["edad"].values >= 65
for nombre, grupo in [("65 años o más", mayores), ("menores de 65", ~mayores)]:
    urgentes = (y_te.values == 1) & grupo
    detectados = int((aviso & urgentes).sum())
    total = int(urgentes.sum())
    print(f"{nombre:15s} → urgentes detectados: {detectados} de {total} ({detectados / max(total, 1):.0%})")

### Qué deberías ver
- **65 años o más:** 41 de 43 (≈ 95 %).
- **Menores de 65:** 78 de 89 (≈ 88 %).

### Conclusión del paso
El modelo detecta algo mejor a los mayores. Con grupos tan pequeños la diferencia es **orientativa**, pero es una comprobación obligatoria: un sistema sanitario no puede ser sistemáticamente peor con un grupo de pacientes.

---
## Paso 9 · La app: ordenar la sala de espera

La función `prioridad(pacientes)`:
1. Recibe una **lista de pacientes** (cada uno es un diccionario con sus signos vitales).
2. La convierte en tabla con **las mismas columnas y en el mismo orden** que el entrenamiento (`[X.columns]`).
3. Calcula la probabilidad de ser urgente con el modelo campeón.
4. Marca **PRIORIDAD** si supera el umbral; si no, **en cola**.
5. Ordena de más a menos probabilidad.

**Nota:** la app **ordena y avisa**, no decide. La última palabra la tiene siempre el personal sanitario.

In [ ]:
def prioridad(pacientes):
    tabla = pd.DataFrame(pacientes)[X.columns]
    tabla["prob_urgente"] = final.predict_proba(tabla)[:, 1].round(2)
    tabla["aviso"] = np.where(tabla["prob_urgente"] >= umbral, "PRIORIDAD", "en cola")
    return tabla.sort_values("prob_urgente", ascending=False)

sala = [dict(edad=70, frecuencia_cardiaca=118, tension_sistolica=95,  saturacion_o2=89, temperatura_c=38.5, dolor_0_10=6, dolor_pecho=1, dificultad_respirar=1, llega_en_ambulancia=1),
        dict(edad=25, frecuencia_cardiaca=78,  tension_sistolica=118, saturacion_o2=99, temperatura_c=36.6, dolor_0_10=2, dolor_pecho=0, dificultad_respirar=0, llega_en_ambulancia=0),
        dict(edad=52, frecuencia_cardiaca=95,  tension_sistolica=130, saturacion_o2=95, temperatura_c=37.0, dolor_0_10=5, dolor_pecho=0, dificultad_respirar=0, llega_en_ambulancia=0)]
prioridad(sala)

### Qué deberías ver
| Paciente | Probabilidad aproximada | Aviso |
|---|---|---|
| 70 años, saturación 89 %, dolor de pecho, dificultad para respirar | ≈ 0,80 | PRIORIDAD |
| 52 años, dolor moderado | ≈ 0,02 | en cola |
| 25 años, sano | ≈ 0,01 | en cola |

### Conclusión del paso
El orden coincide con el sentido común: el paciente con signos de gravedad pasa el primero.

---
## Paso 10 · Ejercicios extra

1. Baja el objetivo a un **recall del 80 %** y súbelo al **95 %**: ¿cuántas falsas alarmas más cuesta cada paso?
2. Mira el recall en **más grupos** (por ejemplo, quien llega en ambulancia y quien no).
3. Escribe qué le dirías a una enfermera para que **se fíe del aviso sin dejar de mirar al paciente**.

**Intenta cada ejercicio antes de ejecutar la solución.** Las cifras pueden variar un poco según la versión de las librerías; lo importante es la conclusión.

### Extra 1 · Recall del 80 %, 90 % y 95 % (bosque, examen)
Repetimos la búsqueda del umbral con tres objetivos de seguridad y, para cada uno, contamos en el examen los urgentes detectados y las falsas alarmas.

**Nota:** el bosque se entrena una sola vez fuera del bucle, porque el modelo no cambia; solo cambia el umbral.

In [ ]:
p_bosque = modelos["Bosque"].fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
extra1 = {}
for objetivo in (0.80, 0.90, 0.95):
    u, prec = umbral_para_recall(modelos["Bosque"], objetivo)
    vn_, fp_, fn_, vp_ = confusion_matrix(y_te, p_bosque >= u).ravel()
    extra1[f"{objetivo:.0%}"] = {"umbral": round(u, 3), "avisos correctos de cada 100": round(prec * 100),
                                 "urgentes detectados": f"{vp_} de {vp_ + fn_}", "falsas alarmas": fp_}
pd.DataFrame(extra1).T

**Conclusión:**

| Recall objetivo | Urgentes detectados | Falsas alarmas |
|---|---|---|
| 80 % | ≈ 110 de 132 | ≈ 127 |
| 90 % | ≈ 119 de 132 | ≈ 364 |
| 95 % | ≈ 127 de 132 | ≈ 654 |

Cada paso **cuesta cada vez más**:
- De 80 % a 90 %: unas **237 falsas alarmas más** para detectar 9 urgentes más.
- De 90 % a 95 %: unas **290 falsas alarmas más** para detectar 8 urgentes más.

Los últimos urgentes son los más caros de detectar. Elegir dónde parar es una **decisión clínica**, no técnica.

### Extra 2 · Recall según llegada en ambulancia (bosque al 90 %)
Separamos el examen según `llega_en_ambulancia` y medimos en cada grupo cuántos urgentes se detectan.

In [ ]:
for valor, nombre in ((1, "llega en ambulancia"), (0, "no llega en ambulancia")):
    grupo = X_te["llega_en_ambulancia"].values == valor
    urgentes = grupo & (y_te.values == 1)
    detectados = int((aviso & urgentes).sum())
    print(f"{nombre:23s}: urgentes detectados {detectados} de {int(urgentes.sum())} ({detectados / max(urgentes.sum(), 1):.0%})")

**Conclusión:** en ambulancia ≈ **33 de 35 (94 %)**; sin ambulancia ≈ **86 de 97 (89 %)**. Resultados parecidos y, otra vez, con grupos pequeños. Merece la pena vigilar a quien llega por su propio pie, porque es donde se escapan más urgentes.

### Extra 3 · Qué decirle a la enfermera
*(Respuesta de redacción: no hay una única respuesta correcta.)*

Escribid tres frases: **qué significa el aviso**, **qué no significa** y **qué hacer si no coincide con lo que ve la enfermera**.

**Nuestra respuesta:**

> …

**Respuesta modelo:**
> *«El aviso sirve para decidir a quién mirar antes, no para dejar de mirar a nadie. Si el modelo dice "en cola" y tú ves algo raro, manda tu criterio. Y si el aviso falla, comunícalo para que podamos revisarlo.»*

---
## Paso 11 · Hoja del equipo

| Campo | Contenido |
|---|---|
| **Equipo** | *(nombres)* |
| **Miniapp** | 9 · ¿Este paciente es urgente? |
| **Fecha** | *(dd/mm/aaaa)* |
| **Datos** | 4 000 pacientes inventados · 13 % urgentes · separación 75/25 estratificada |
| **Modelos comparados** | Tonto, logística, árbol explicable, bosque, boosting |
| **Modelo campeón** | Bosque aleatorio (300 árboles) |
| **Por qué ese modelo** | Mejor AUC (≈ 0,87) y AP (≈ 0,64), y menos falsas alarmas al 90 % de recall |
| **Alternativa explicable** | Árbol de profundidad 4 (AUC ≈ 0,80), se puede leer como reglas |
| **Criterio del umbral** | Detectar al menos el 90 % de los urgentes (recall 0,90) |
| **Resultado en el examen** | 119 de 132 urgentes detectados · 364 falsas alarmas de 868 · AUC ≈ 0,89 |
| **Comparación con el modelo tonto** | El tonto acierta el 86,8 % pero detecta **0** urgentes |
| **Revisión por grupos** | ≥ 65 años: 95 % · < 65 años: 88 % (grupos pequeños, orientativo) |

### Tarjeta de la app
- La app decide **a quién atender antes** a partir de **dolor, pulso, saturación y respiración**.
- Antes de modelar hubo que **fijar el recall en el 90 %** y aceptar las falsas alarmas.
- Detecta **119 de 132 urgentes** en el examen (con 364 falsas alarmas), frente a **0** del modelo tonto.
- **No debe usarse para decidir sin una persona**: no es una herramienta médica.

### Decisión del equipo: ¿explicable o potente?
*(Explicad vuestra elección en 2–3 frases.)*

> …

### Límites
- Los datos son **inventados**.
- Un modelo opaco es difícil de **explicar y de auditar**, aunque acierte más.
- Con pocos urgentes en el examen, las cifras **oscilan**.

### Lo que hemos aprendido
- *(escribid 2–3 ideas con vuestras palabras)*

La celda siguiente imprime **vuestras cifras reales** para comprobar o completar la tabla:

In [ ]:
u_arbol, p_arbol = resultados_umbral["Árbol explicable (profundidad 4)"]
u_bosq, p_bosq = resultados_umbral["Bosque"]
resumen = pd.Series({
    "Pacientes / urgentes": f"{len(df)} / {int(y.sum())} ({y.mean():.1%})",
    "Precisión global del modelo tonto (CV)": f"{tabla.loc['Nadie es urgente (tonto)', 'precisión global']:.1%}",
    "AUC / AP árbol explicable (CV)": f"{tabla.loc['Árbol explicable (profundidad 4)', 'AUC']:.3f} / {tabla.loc['Árbol explicable (profundidad 4)', 'AP']:.3f}",
    "AUC / AP bosque (CV)": f"{tabla.loc['Bosque', 'AUC']:.3f} / {tabla.loc['Bosque', 'AP']:.3f}",
    "Avisos correctos de cada 100 (árbol / bosque)": f"{p_arbol * 100:.0f} / {p_bosq * 100:.0f}",
    "Campeón y umbral": f"{campeon_nombre} · {umbral:.2f}",
    "Urgentes detectados en examen": f"{vp} de {vp + fn}",
    "Falsas alarmas en examen": f"{fp} de {vn + fp}",
    "AUC en examen": f"{roc_auc_score(y_te, proba):.3f}",
}, name="Valor")
resumen.to_frame()

---
## Paso 12 · Prueba

### A) Formulario para probar un paciente
En Colab, a la derecha de la celda aparecen **casillas y deslizadores**. Cambia los valores y vuelve a ejecutar la celda para ver la prioridad del paciente.

**Nota:** las líneas con `#@param` son las que Colab convierte en formulario. Fuera de Colab funcionan como variables normales.

In [ ]:
#@title Revisar un paciente
edad = 60  #@param {type:"slider", min:0, max:100, step:1}
frecuencia_cardiaca = 105  #@param {type:"slider", min:40, max:150, step:1}
tension_sistolica = 110  #@param {type:"slider", min:60, max:200, step:1}
saturacion_o2 = 92  #@param {type:"slider", min:85, max:100, step:1}
temperatura_c = 37.8  #@param {type:"number"}
dolor_0_10 = 5  #@param {type:"slider", min:0, max:10, step:1}
dolor_pecho = 0  #@param [0, 1]
dificultad_respirar = 1  #@param [0, 1]
llega_en_ambulancia = 0  #@param [0, 1]

paciente = dict(edad=edad, frecuencia_cardiaca=frecuencia_cardiaca, tension_sistolica=tension_sistolica,
                saturacion_o2=saturacion_o2, temperatura_c=temperatura_c, dolor_0_10=dolor_0_10,
                dolor_pecho=dolor_pecho, dificultad_respirar=dificultad_respirar,
                llega_en_ambulancia=llega_en_ambulancia)
prioridad([paciente])

### B) Sala de espera de prueba
Simulamos una sala con seis pacientes de perfiles distintos. Esperamos que los pacientes con signos de gravedad (saturación baja, dolor de pecho, dificultad para respirar) **salgan arriba** y los sanos, **abajo**.

In [ ]:
sala_prueba = pd.DataFrame([
    dict(paciente="A · Joven con esguince",          edad=22, frecuencia_cardiaca=80,  tension_sistolica=120, saturacion_o2=99, temperatura_c=36.7, dolor_0_10=4, dolor_pecho=0, dificultad_respirar=0, llega_en_ambulancia=0),
    dict(paciente="B · Niño con fiebre",             edad=6,  frecuencia_cardiaca=110, tension_sistolica=100, saturacion_o2=97, temperatura_c=38.9, dolor_0_10=3, dolor_pecho=0, dificultad_respirar=0, llega_en_ambulancia=0),
    dict(paciente="C · Mayor con dolor de pecho",    edad=78, frecuencia_cardiaca=112, tension_sistolica=90,  saturacion_o2=91, temperatura_c=36.9, dolor_0_10=7, dolor_pecho=1, dificultad_respirar=1, llega_en_ambulancia=1),
    dict(paciente="D · Adulto con migraña",          edad=40, frecuencia_cardiaca=82,  tension_sistolica=135, saturacion_o2=98, temperatura_c=36.8, dolor_0_10=6, dolor_pecho=0, dificultad_respirar=0, llega_en_ambulancia=0),
    dict(paciente="E · Asmático con crisis",         edad=35, frecuencia_cardiaca=120, tension_sistolica=125, saturacion_o2=90, temperatura_c=37.1, dolor_0_10=3, dolor_pecho=0, dificultad_respirar=1, llega_en_ambulancia=0),
    dict(paciente="F · Control rutinario",           edad=50, frecuencia_cardiaca=72,  tension_sistolica=128, saturacion_o2=98, temperatura_c=36.5, dolor_0_10=1, dolor_pecho=0, dificultad_respirar=0, llega_en_ambulancia=0),
])
orden = prioridad(sala_prueba.drop(columns="paciente").to_dict("records"))
orden.insert(0, "paciente", sala_prueba.loc[orden.index, "paciente"].values)
orden[["paciente", "edad", "saturacion_o2", "dolor_pecho", "dificultad_respirar", "prob_urgente", "aviso"]]

### Qué deberías ver
- Arriba, el **asmático con crisis** y el **mayor con dolor de pecho** (≈ 0,75): PRIORIDAD.
- En medio, el **niño con fiebre** (≈ 0,32).
- Abajo, el esguince y la migraña (≈ 0,02): en cola.

**Conclusión:** el orden tiene sentido clínico. Fíjate en que el **control rutinario** (≈ 0,08) también recibe aviso: como el umbral es muy bajo (≈ 0,05) para no dejar escapar urgentes, el modelo avisa incluso con probabilidades pequeñas. Es el precio del recall del 90 % que vimos en el paso 6.

### C) Comprobaciones automáticas
Pequeñas verificaciones de que el cuaderno ha funcionado correctamente. Cada línea indica **OK** o **FALLO**.

In [ ]:
def check(condicion, mensaje):
    print(("OK    " if condicion else "FALLO ") + mensaje)
    return bool(condicion)

ok = all([
    check(len(df) == 4000, "El dataset tiene 4 000 pacientes"),
    check(abs(y_tr.mean() - y_te.mean()) < 0.005, "Entrenamiento y examen tienen la misma proporción de urgentes"),
    check(tabla.loc["Nadie es urgente (tonto)", "AUC"] == 0.5, "El modelo tonto tiene AUC 0,50"),
    check(tabla.loc["Bosque", "AUC"] > tabla.loc["Árbol explicable (profundidad 4)", "AUC"], "El bosque supera al árbol en AUC"),
    check(p_bosq > p_arbol, "Al 90 % de recall, el bosque da menos falsas alarmas que el árbol"),
    check(vp / (vp + fn) >= 0.85, "En el examen se detecta al menos el 85 % de los urgentes"),
    check(orden.iloc[0]["paciente"].startswith(("C", "E")), "El primer paciente de la sala es uno con signos de gravedad"),
    check(orden.iloc[-1]["aviso"] == "en cola", "El último paciente de la sala queda en cola"),
])
print("\nTodas las pruebas superadas." if ok else "\nRevisa las pruebas marcadas como FALLO.")

---
## Conclusiones finales

1. **La precisión global engaña** cuando una clase es rara: el modelo que dice «nadie es urgente» acierta el 86,8 % y no sirve para nada. Hay que mirar **AUC, AP y la matriz de confusión**.
2. **El umbral se elige pensando en el coste de los errores.** Aquí lo más grave es que se escape un urgente, así que fijamos un **recall del 90 %** y aceptamos las falsas alarmas que eso conlleva.
3. **Explicable frente a potente:** el bosque da más avisos útiles (≈ 24 de cada 100 frente a ≈ 15 del árbol), pero el árbol se puede **explicar y auditar**. Es una decisión de criterio, no solo de nota.
4. **Cada punto extra de seguridad cuesta cada vez más falsas alarmas** (del 90 % al 95 % de recall, casi el doble).
5. **Hay que revisar por grupos** (edad, ambulancia): un buen promedio puede esconder peores resultados en un grupo concreto.
6. **El modelo avisa, la persona decide.** Esta herramienta sirve para ordenar la atención, nunca para sustituir el criterio sanitario.